In [2]:
import pandas as pd, numpy as np
df = pd.read_csv("../data/raw_nifty.csv", parse_dates=["Date"])

print("Range:", df["Date"].min(), "to", df["Date"].max(), "| rows:", len(df))
print("Duplicate dates:", df["Date"].duplicated().sum())
print("Sorted ascending:", df["Date"].is_monotonic_increasing)
print("Missing values:\n", df.isna().sum())

Range: 2007-09-17 00:00:00 to 2026-09-23 00:00:00 | rows: 4665
Duplicate dates: 0
Sorted ascending: True
Missing values:
 Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64


In [3]:
ohlc = ["Open", "High", "Low", "Close"]
print("Non-positive prices:", (df[ohlc] <= 0).any(axis=1).sum())
print("High < Low:", (df["High"] < df["Low"]).sum())
print("Open/Close outside High-Low:",
      ((df["High"] < df[["Open","Close"]].max(axis=1)) |
       (df["Low"] > df[["Open","Close"]].min(axis=1))).sum())
print("Zero-range days (High == Low):", (df["High"] == df["Low"]).sum())
print("Adj Close != Close:", (df["Adj Close"] != df["Close"]).sum())

Non-positive prices: 0
High < Low: 0
Open/Close outside High-Low: 0
Zero-range days (High == Low): 0
Adj Close != Close: 0


In [4]:
df["gap_days"] = df["Date"].diff().dt.days
print(df.loc[df["gap_days"] > 4, ["Date", "gap_days"]])

df["ret"] = df["Close"].pct_change()
print(df.loc[df["ret"].abs() > 0.06, ["Date", "Close", "ret"]])
print(df["ret"].describe())
print("Days with ret <= -2%:", (df["ret"] <= -0.02).sum())
print(df.tail(3))

           Date  gap_days
128  2008-03-24       5.0
392  2009-05-04       5.0
555  2009-12-29       5.0
1114 2012-04-09       5.0
1723 2014-10-07       6.0
1734 2014-10-27       5.0
1840 2015-04-06       5.0
2079 2016-03-28       5.0
2092 2016-04-18       5.0
2174 2016-08-16       5.0
2574 2018-04-02       5.0
3569 2022-04-18       5.0
           Date        Close       ret
86   2008-01-21  5208.799805 -0.087024
88   2008-01-23  5203.399902  0.062070
90   2008-01-25  5383.350098  0.069515
264  2008-10-10  3279.949951 -0.066512
265  2008-10-13  3490.699951  0.064254
274  2008-10-24  2584.000000 -0.122029
276  2008-10-29  2697.050049  0.068477
277  2008-10-31  2885.600098  0.069910
284  2008-11-11  2938.649902 -0.066577
320  2009-01-07  2920.399902 -0.061809
402  2009-05-18  4323.149902  0.177441
3050 2020-03-12  9590.150391 -0.083019
3052 2020-03-16  9197.400391 -0.076121
3057 2020-03-23  7610.250000 -0.129805
3059 2020-03-25  8317.849609  0.066247
3066 2020-04-07  8792.200195  0.087632

In [5]:
raw = pd.read_csv("../data/raw_nifty.csv", parse_dates=["Date"])
clean = raw[["Date", "Open", "High", "Low", "Close"]]
clean = clean[clean["Date"] < "2026-09-23"]   # drop today's possibly incomplete bar
clean.to_csv("../data/clean_nifty.csv", index=False)
print(clean.shape, clean["Date"].min(), clean["Date"].max())

(4664, 5) 2007-09-17 00:00:00 2026-09-21 00:00:00


Cleaning log
- No duplicates, missing values or invalid OHLC found.
- Kept all extreme moves (2008, 2009-05-18, 2020): real events.
- Dropped last row (2026-09-23): Volume 0, possibly incomplete bar.
- 2026-09-22 absent: unverified single-day gap, left as is.
- Removed Adj Close (identical to Close) and Volume (unreliable for index).
- Raw file untouched; cleaned file is data/clean_nifty.csv.